# 1. Building the project and the hydrological model

Workflow: **`build_model.smk`** (WF1)

<div style="border-left:4px solid #999;padding:0.5em 0.9em;background:#f6f6f6">
<b>Source only &mdash; outputs are not committed.</b><br>
Run the notebook to see results, or read a rendered copy published as an
Artifact. Outputs are cleared on commit
(<code>dev/scripts/notebook_outputs.py --strip</code>) and their absence is
enforced by <code>tests/test_notebook_outputs.py</code> on both CI legs.<br>
Why: a notebook carrying figures embeds them as base64 and does not
delta-compress, so every edit added megabytes to history permanently &mdash;
these three were 8.8 MB, and one rename sweep that rewrote three short
strings inside them cost a 7.1 MB push.
</div>

## What this workflow does

WF1 turns a point on a map into a runnable, forced Wflow-SBM model, plus a first
read on how well that model reproduces observed discharge. In order:

1. **Delineates the basin** from the region you specify, along with the
   subcatchments and gauge locations inside it.
2. **Extracts a historical climate store** for that basin from a global dataset
   (ERA5 by default), and plots it.
3. **Prepares the thematic maps** &mdash; land cover, leaf area index, soil
   properties &mdash; that the model is parameterised from.
4. **Builds the Wflow-SBM model** with hydromt, adds reservoirs, lakes and
   glaciers, and declares which state variables to write out.
5. **Adds climate forcing** for the simulation window and **runs Wflow once**.
6. **Evaluates the run** against observed discharge and plots the result.

This is a *rapid deployment*: the model is parameterised from global data and is
not locally calibrated. The evaluation step is what tells you how far to trust
it &mdash; a question worth settling before the stress test in notebook 3, since
that notebook inherits whatever this one produces.

Nothing here is scenario-driven. The perturbations come later; WF1 builds the
reference they are applied to.

## Setup

In [ ]:
# Standard library, plus the few third-party names the results cells need.
import json
import os
import subprocess
from pathlib import Path

import pandas as pd
import yaml
from IPython import display

In [ ]:
# Work from the repository root.
#
# Snakemake resolves every path in a config relative to the directory it is
# invoked from, and the shipped configs are written against the repo root.
# Rather than hardcoding an install path -- which is what made the previous
# version of this notebook unrunnable for anyone but its author -- walk up from
# wherever the kernel started until the Snakefiles come into view.
def find_repo_root(marker: str = "build_model.smk") -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(
        f"could not find {marker} at or above {Path.cwd()}; "
        "start the kernel inside the blueearth_cst checkout"
    )


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
print(REPO_ROOT)

In [ ]:
# Snakemake runs take minutes, so stream the output rather than buffering it to
# the end of the cell.
#
# `check` raises on a nonzero exit. Without it a failed Snakemake call printed
# its error into the cell output and the cell still rendered as SUCCESSFUL, so
# the first symptom was a later `display.Image` raising on a file that was never
# written -- an error pointing at the wrong place, in a document whose reader is
# the least equipped to tell the difference. Pass `check=False` where a nonzero
# exit is expected, as `snakemake --unlock` gives when there is no lock.
def run(command: str, *, check: bool = True) -> int:
    print("$ " + command)
    print()
    with subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        shell=True,
        stderr=subprocess.STDOUT,
        bufsize=1,
        close_fds=True,
    ) as process:
        for line in iter(process.stdout.readline, b""):
            print(line.rstrip().decode("utf-8", errors="replace"))
    if check and process.returncode != 0:
        raise RuntimeError(
            f"command failed with exit code {process.returncode}: {command}"
        )
    return process.returncode

## The settings file

Everything a run does is decided by one YAML file passed as `--configfile`. The
toolbox ships seed configs beside the project they write into, under
`test_case/`:

| File | Project | Use it for |
|---|---|---|
| `project_config_rapid.yml` | `test_case/test_rapid` | anything you want to watch execute &mdash; the default |
| `project_config_baseline.yml` | `test_case/test_local` | runs whose numbers get quoted or fingerprinted |
| `project_config_wf2_fast.yml` | `test_case/test_dev` | WF2 code iteration only |

To start your own project, copy `config/templates/project_config.template.yml` and
edit it. If the copy lives under `test_case/`, keep the `project_config_` prefix:
the repository's `.gitignore` un-ignores exactly that glob, and a name outside it
is silently untracked.

This notebook runs the rapid config.

In [ ]:
# The settings file this notebook runs against, and the paths derived from it
# that every later cell needs.
CONFIG = "test_case/project_config_rapid.yml"

cfg = yaml.safe_load(Path(CONFIG).read_text(encoding="utf-8"))
PROJECT_DIR = Path(cfg["project"]["project_dir"])
PROJECT_NAME = cfg.get("project_name") or PROJECT_DIR.name
print("config      :", CONFIG)
print("project_dir :", PROJECT_DIR)

In [ ]:
# The settings file, in full. It is commented in place -- those comments are the
# documentation for each key, which is why this cell prints the shipped file
# rather than re-authoring a copy that could drift away from it.
print(Path(CONFIG).read_text(encoding="utf-8"))

### Reading the settings file

Three top-level sections, and the split between them is meaningful.

**`project:`** &mdash; where outputs go and which data catalogs to read.
`project_dir` is the root of everything the run writes; in a real project it
lives *outside* the repository. `catalog` points at a hydromt data catalog
under `config/catalogs/`. Data is registered in a catalog and handed to hydromt
with `-d`; it is never hardcoded in a rule, so pointing a project at different
inputs is a config edit rather than a code edit.

**`basin:`, `climate:`, `model:`** &mdash; the facts more than one workflow
reads, and therefore the facts they must agree on.
`basin.region` is a hydromt region expression, `basin.resolution` the model grid
in degrees, `historical_window` the period the climate store covers, and
`climate.selected` the source dataset. WF2 and WF3 read these same keys, which is
what keeps the three workflows describing one basin rather than three.

`basin.region` accepts any of:

- a point anywhere in a basin that drains to the sea &mdash; `{'basin': [x, y]}`
- a subbasin outlet &mdash; `{'subbasin': [x, y], 'uparea': 100}`
- a bounding box &mdash; `[xmin, ymin, xmax, ymax]`
- a geometry file &mdash; `{'subbasin': 'myregion.shp'}`

Coordinates are EPSG:4326. For a point, <http://bboxfinder.com/> is the quickest
way to get one.

**`workflows:`** &mdash; one block per workflow, each with its own `enabled:`
flag. WF1 reads `workflows.build_model`. Two keys there are worth pausing on:

- `simulation_window` is the period Wflow actually *runs*, a subset of
  `climate.window`. WF1's runtime scales with it directly.
- `observations` is optional. Without it the model is still built and
  run &mdash; you simply get no performance metrics, and therefore no basis for
  judging the stress-test results downstream.

## Input file formats

Two optional CSVs shape what WF1 produces. Both are shown here rather than
specified, because a schema is easier to copy than to read.

**`basin.output_locations`** &mdash; the locations you want output at.
Required columns are `station_name`, `x`, `y`. Optional: `location_role`
(`control` or `observation`, default `control`) and `wflow_id`. Points are
snapped to the river network during delineation, so a coordinate that sits
slightly off the channel still works.

In [ ]:
pd.read_csv(cfg["basin"]["output_locations"])

**`workflows.build_model.observations`** &mdash; observed
discharge in m&sup3;/s. **Semicolon-separated**, with a `time` column and one
column per station.

The column headers are **`wflow_id` values, not station names**. Those ids are
assigned during the model build, so the sequence is: run once, read the ids back
from the location registry (shown in the results section), then name your
observation columns to match.

In [ ]:
obs = pd.read_csv(
    cfg["workflows"]["build_model"]["observations"],
    sep=";",
    index_col=0,
    parse_dates=True,
)
print(obs.shape)
obs.head()

## The job graph

Rendering the DAG before running is a cheap check that the config was parsed the
way you intended. A mistyped optional key usually shows up as a missing branch
rather than as an error.

In [ ]:
# Render the job graph. `scripts/plot_workflow_dag.py` wraps `snakemake --dag`
# and pipes it through graphviz, so the image lands inside the project's own
# logs/ rather than in whatever directory the shell happened to be in:
#
#     <project_dir>/logs/dag/<project_name>_wf<N>[_<experiment>]_dag.png
#
# Reading the graph before running is the cheapest way to see whether the config
# was understood the way you meant it.
run(f"python scripts/plot_workflow_dag.py -s build_model.smk --configfile {CONFIG}")

In [ ]:
DAG_PNG = PROJECT_DIR / "logs" / "dag" / f"{PROJECT_NAME}_wf1_dag.png"
display.Image(str(DAG_PNG))

### The rules, and the config keys that tune them

Rules are numbered, and the numbers are stable &mdash; they appear in the run
log, in the benchmark table, and in the graph above. The gaps are deliberate:
rules are never renumbered to close one, because the number is a reference other
documents cite.

| Rule | What it does | Config keys that tune it |
|---|---|---|
| **1.01** `snapshot_config` | records the effective config under `<project_dir>/config/runs/build_model/`, so a result can be traced to the settings that produced it | `project.project_dir` |
| **1.02** `delineate_region` | resolves the region expression into a basin polygon | `basin.region` |
| **1.03** `delineate_spatial_units` | subcatchments, rivers, gauge locations, and the location registry | `basin.region`, `basin.resolution`, `basin.output_locations` |
| **1.04** `extract_historical_climate` | clips the global climate dataset to the basin, stored as `<source>_<window>` | `climate.selected`, `climate.window`, `project.data_sources` |
| **1.05** `plot_climate_source` | the `source_*.png` figures of that store | &mdash; draws what 1.04 built |
| **1.06** `prepare_spatial_maps` | folds land cover, LAI and soil properties onto the model grid | `project.catalog` |
| **1.07** `build_wflow_model` | the hydromt `build` &mdash; where Wflow-SBM is parameterised | `workflows.build_model.engine.build_config`, `basin.resolution` |
| **1.08** `add_reservoirs_lakes_glaciers` | hydromt `update` for the waterbody layers | `workflows.build_model.engine.waterbodies_config` |
| **1.09** `declare_wflow_outputs` | writes the output section of `wflow_sbm.toml` | `workflows.build_model.model.outvars` |
| **1.10** `add_climate_forcing` | builds the forcing netCDF for the run period | `climate.selected`, `workflows.build_model.simulation_window` |
| **1.11** `write_outlet_index` | maps station names onto the `wflow_id`s the model assigned | `basin.output_locations` |
| **1.12** `plot_basin_map` | `basin_area.png` plus the thematic map family | &mdash; |
| **1.13** `plot_forcing` | the `forcing_*.png` maps and series | &mdash; |
| **1.14** `run_wflow` | runs Wflow (Julia) | `workflows.build_model.simulation_window`, `advanced_settings.runtime.julia_threads` |
| **1.14b** `export_wflow_tables` | splits raw output into `output_q.csv`, `output_aet.csv`, `output_gwr.csv` | `workflows.build_model.model.outvars` |
| **1.15** `plot_wflow_evaluation` | performance metrics and hydrographs against observations | `workflows.build_model.observations` |
| **1.15b** `write_run_metadata` | a provenance sidecar recording the config digest the outputs came from | &mdash; |
| **1.16** `gather_benchmarks`, **1.17** `gather_logs` | merge the per-rule parts into one table and one log | &mdash; |

Three keys move the *cost* of a run more than anything else: `simulation_window`
(Wflow runtime, linear), `historical_window` (how much climate data is
extracted), and `basin.resolution` (grid cells, quadratic).

## Running

Snakemake locks its working directory while a run is in progress, and does not
always release the lock if a run is killed. Unlocking first is harmless when
there is nothing to unlock.

In [ ]:
run(f"snakemake --unlock -s build_model.smk --configfile {CONFIG}", check=False)

A dry run lists the jobs that *would* execute, and why, without running
them. On a project that is already up to date it reports nothing to do &mdash;
which is itself the answer to "did my config edit invalidate anything?".

In [ ]:
run(f"snakemake all -n -s build_model.smk --configfile {CONFIG}")

And the real run. `-c 3` gives Snakemake three cores to schedule with;
raise it if you have them, since several branches of this graph are independent.

Expect this to take a while on a cold project &mdash; the hydromt build and the
Wflow run dominate the wall clock.

In [ ]:
run(f"snakemake all -c 3 -s build_model.smk --configfile {CONFIG}")

## Results

In [ ]:
# What the run wrote. Noisy internal branches are skipped so the shape of the
# tree stays readable.
def show_tree(root: Path, skip=(".snakemake", "_parts", "_work"), max_files=8):
    root = Path(root)
    for path, dirs, files in os.walk(root):
        dirs[:] = sorted(d for d in dirs if d not in skip)
        rel = Path(path).relative_to(root)
        print(root.name + "/" if str(rel) == "." else f"{root.name}/{rel}")
        shown = sorted(f for f in files if not f.endswith(".xml"))
        for name in shown[:max_files]:
            print("  - " + name)
        if len(shown) > max_files:
            print(f"  ... {len(shown) - max_files} more")


show_tree(PROJECT_DIR)

### The basin

`basin_area.png` is the orientation figure: catchment outline, river network,
elevation, gauge locations, and a locator inset.

Check three things here before looking at any number &mdash; that the outline is
the basin you meant, that the gauge points sit *on* the river rather than beside
it, and that the drainage network is connected. A point that snapped to the
wrong tributary produces a model that runs perfectly well and answers a
different question than the one you asked.

In [ ]:
display.Image(str(PROJECT_DIR / "data" / "spatial" / "plots" / "basin_area.png"))

The thematic maps beside it &mdash; land cover, soil texture, depth to
bedrock, leaf area index &mdash; are what Wflow-SBM is parameterised from. They
are worth a glance mainly as a check on the data catalog: a soil map that is
uniform across the basin usually means the global product has no real resolution
here, and the model's spatial variability is coming from topography alone.

In [ ]:
display.Image(str(PROJECT_DIR / "data" / "spatial" / "plots" / "land_cover.png"))

### The forcing

Precipitation, temperature and PET as the model actually saw them. The annual
series is the quickest check that the climate store covers the simulation window
without gaps.

In [ ]:
FORCING_PLOTS = PROJECT_DIR / "models" / "hydrology" / "wflow" / "forcing" / "plots"
display.display(
    display.Image(str(FORCING_PLOTS / "forcing_precip_annual.png")),
    display.Image(str(FORCING_PLOTS / "forcing_temp_annual.png")),
)

### Station identity

The location registry is the bridge between the names you supplied in
`output_locations` and the `wflow_id`s the model assigned &mdash; the ids your
observation file has to use. It also records where each point *snapped to*,
which is the first thing to check when one station's hydrograph looks wrong.

In [ ]:
registry = pd.read_csv(PROJECT_DIR / "data" / "spatial" / "location_registry.csv")
registry[
    [
        "station_name",
        "wflow_id",
        "location_role",
        "original_x",
        "original_y",
        "snapped_x",
        "snapped_y",
    ]
]

### Model performance

`performance_metrics.csv` holds KGE, NSE and their variants per station, at
daily and monthly aggregation.

In [ ]:
metrics = pd.read_csv(
    PROJECT_DIR / "models" / "hydrology" / "wflow" / "evaluation" / "performance_metrics.csv"
)
metrics.round(3)

**How to read this.** These are uncalibrated results from global data, so
the question is not "is it good" but "is it good enough for the decision this
assessment supports".

- **KGE above ~0.7** means water balance, variability and timing are all roughly
  right at that station. Stress-test results there are worth interpreting in
  absolute terms.
- **KGE ~0.4&ndash;0.7** usually means one component is off, most often the
  mean &mdash; a hydrograph with the right shape at the wrong level. *Relative*
  changes across the stress-test grid can still be informative when absolute
  values are not.
- **KGE below ~0.4** is a station to be careful with. In this rendered run the
  smallest tributary sits here, which is the usual pattern: global forcing on a
  ~1 km grid resolves the main stem far better than a headwater.
- **Monthly better than daily** is normal, and means the model has the seasonal
  cycle but not individual events. For a stress test built on annual and
  seasonal indicators that is often acceptable; for a flood application it is
  not.

Read the metrics *per station*. A good outlet number can hide a tributary that is
not being simulated at all, and averaging across stations is exactly how that
gets missed.

The hydrographs make the same point visually &mdash; one per station,
named by `wflow_id`, simulated against observed.

In [ ]:
hydrographs = sorted(
    (PROJECT_DIR / "models" / "hydrology" / "wflow" / "evaluation" / "plots").glob(
        "hydrograph_*.png"
    )
)
display.display(*[display.Image(str(p)) for p in hydrographs])

### Provenance

`run_metadata.json` records the digest of the config the outputs were produced
from. It is how a later workflow can tell that the model it is about to reuse
was built from settings that have since changed &mdash; the check WF3 performs
before it will run.

In [ ]:
run_metadata = json.loads(
    (
        PROJECT_DIR / "models" / "hydrology" / "wflow" / "evaluation" / "run_metadata.json"
    ).read_text(encoding="utf-8")
)
print(json.dumps(run_metadata, indent=2))

---

## Next

**[2. Climate projections](<Climate projections.ipynb>)** &mdash; CMIP6 change
factors for this basin, which situate the stress-test grid in projection space.

Then **[3. Climate stress test](<Climate Stress Test.ipynb>)**, which reuses the
model built here. Run the three in order: WF3 checks that the WF1 model it finds
matches the settings it was told about, and refuses to run against a stale
one.